# Parameter Efficient Fine-Tuning (PEFT) using LoRA 🚀

**Is notebook mein hum seekhenge:**
1. Kaise **LoRA (Low-Rank Adaptation)** ka use karke LLM (TinyLlama) ke limited adapter weights ko train kiya jata hai.
2. Kaise bitsandbytes load quantization parameters ko MPS / Mac par safety ke saath bypass kiya jata hai.
3. Kaise fine-tuned checkpoints load karke evaluation prompts se output text generate karte hain.

In [1]:
# Dependencies check aur install
!uv pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF --quiet

In [2]:
from datasets import Dataset, load_dataset

## 1. Dataset Loading & Preparation (डेटा की तैयारी) 📂

Hamein LoRA training ke liye bhi data preprocess karna hoga. Yahan wahi PDF extraction logic setup hai.

In [3]:
# dataset = load_dataset("HuggingFaceFW/fineweb")
# pubmed = load_dataset("ncbi/pubmed")
# dataset = load_dataset("datajuicer/the-pile-oubmed-abstracts-refined-by-data-juicer")
# dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")

# owt = load_dataset("Skylion007/openwebtext")
# ds = load_dataset("armanc/scientific_papers")


### Dataset from Hub Option 🌐

In [4]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

In [5]:
print(dataset)
print(dataset[0])
print(dataset[-1])

Dataset({
    features: ['text'],
    num_rows: 2119719
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
{'text': 'Once upon a time, there was an adorable little cat named Kitty. Kitty loved to polish her toy car with a soft cloth. One sunny day, she decided to take her shiny car to the park.\n\nAt the park, she met a friendl

### Custom Metformin PDF Processing 📄

In [6]:
# PDF processing helpers load karte hain
import fitz

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    
    return text_blocks


pdf_texts = extract_text_from_pdf("Metformin.pdf")

print(pdf_texts)
print(len(pdf_texts[0]))

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [7]:
import re

def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:
                paragraphs.append(clean)
    
    return paragraphs

In [8]:
# PDF blocks convert to list of clean text paragraphs
paragraphs = split_paragraphs(pdf_texts)
paragraphs

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA redu

In [9]:
data = [{"text": p} for p in paragraphs]
data

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

In [10]:
# HF Dataset wrap create karte hain
dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

## 2. LoRA Adapter Configuration & Training 🛠️

Chaliye model load karte hain, use wrap karte hain LoRA parameters ke saath aur training launch karte hain.

In [11]:
# Pehle unused CPU/GPU (MPS) variables aur cache clear karte hain memory management ke liye
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [12]:
# LoRA integration modules aur wrapper classes import karte hain
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [13]:
# Model repository select karte hain
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [14]:
# Tokenizer initialization
tokenizer = AutoTokenizer.from_pretrained(model)

In [15]:
# Padding setting coordinate karte hain
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
# Tokenize map function creation

def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [17]:
# Tokenized inputs ready karte hain training data feeding ke liye
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [18]:
# Model loading: TinyLlama ko directly bfloat16 mein load karte hain (8-bit bitsandbytes MPS par crash hota hai aur NaN dega)
# Load model directly in bfloat16 (8-bit quantization is unstable on MPS)
model = AutoModelForCausalLM.from_pretrained(
    model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:
# LoRA configuration specify karte hain (Rank r=8 aur projections adapters set karte hain)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [20]:
# Base model ke trainable parameters adjust karke check karte hain
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)

In [21]:
# LoRA Training args setup. bf16=True MPS compatibility ke liye critical hai
args = TrainingArguments(
    output_dir="./tinyllama-lora-non-instruction",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [22]:
# Model Trainer setup configuration
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [23]:
# LoRA adapters training execute karte hain!
trainer.train()

/Users/kevin/Desktop/engineer/LLMs-from-scratch/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=5, training_loss=9.378365325927735, metrics={'train_runtime': 12.5303, 'train_samples_per_second': 1.596, 'train_steps_per_second': 0.399, 'total_flos': 63629646888960.0, 'train_loss': 9.378365325927735, 'epoch': 5.0})

In [24]:
# Trained weights model checkpoint aur tokenizer local store save karte hain
# Save & test the model
trainer.save_model("./tinyllama-lora-non-instruction")
tokenizer.save_pretrained("./tinyllama-lora-non-instruction")

('./tinyllama-lora-non-instruction/tokenizer_config.json',
 './tinyllama-lora-non-instruction/tokenizer.json')

In [25]:
# Training complete hone ke baad saved checkpoint path set karte hain
model_path = "tinyllama-lora/checkpoint-5"

In [26]:
# Model loading: TinyLlama ko directly bfloat16 mein load karte hain (8-bit bitsandbytes MPS par crash hota hai aur NaN dega)
trained_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")

OSError: tinyllama-lora/checkpoint-5 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# Test prompt defining medical drug context
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [ ]:
# Prompt tokenization aur device setting (MPS pe text push karte hain)
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

In [ ]:
# Inference execution: max_length config set to None to avoid validation warnings during generation
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# Final text output decode aur decode display
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was better than either drug alone, and the new trial is designed to evaluate the impact of this combination.
"Atrial fibrillation is a common heart condition in which an irregular heartbeat causes blood to flow backward through the atrium, the upper chamber of the heart," says Dr. Lurie. "The condition increases the risk of stroke and other serious cardiac problems, so it's important to treat patients who have it."
Dr. Lurie
